# Este pipeline se encarga de calcular el RUL (Vida Util Restante) de ambos datasets

In [25]:
# IMPORTS
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

In [3]:
# Sesión Spark
spark = (
    SparkSession.builder
    .appName("TFM_CMAPPSS_FD001_RUL")
    .getOrCreate()
)
spark.conf.set("spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version", "2")
spark.conf.set("spark.hadoop.mapreduce.fileoutputcommitter.cleanup-failures.ignored", "true")

In [18]:
# Importamos datasets
df_train = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("sep", ",")
    .csv("/opt/spark-data/interim/TRAIN_FD001_CLEAN.csv")
)
df_test = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("sep", ",")
    .csv("/opt/spark-data/interim/TEST_FD001_CLEAN.csv")
)
df_rul= (
    spark.read
    .option("sep", " ")
    .option("inferSchema", "true")
    .csv("/opt/spark-data/raw/RUL_FD001.txt")
    .withColumnRenamed("_c0", "RUL_final")
    .withColumn("unit_number", F.monotonically_increasing_id() + 1)
    .select("unit_number", "RUL_final")
)

df_rul.show()

+-----------+---------+
|unit_number|RUL_final|
+-----------+---------+
|          1|      112|
|          2|       98|
|          3|       69|
|          4|       82|
|          5|       91|
|          6|       93|
|          7|       91|
|          8|       95|
|          9|      111|
|         10|       96|
|         11|       97|
|         12|      124|
|         13|       95|
|         14|      107|
|         15|       83|
|         16|       84|
|         17|       50|
|         18|       28|
|         19|       87|
|         20|       16|
+-----------+---------+
only showing top 20 rows



In [19]:
print("Columnas train:", len(df_train.columns),"Filas:",df_train.count())
print("Columnas test:", len(df_test.columns),"Filas:",df_test.count())
print("Columnas rul:", len(df_rul.columns),"Filas:",df_rul.count())


Columnas train: 18 Filas: 20631
Columnas test: 18 Filas: 13096
Columnas rul: 2 Filas: 100


## Calculo el RUL para TRAIN (último ciclo del motor - ciclo actual)

In [22]:
# Cálculo del RUL para TRAIN
ventana_unit = Window.partitionBy("unit_number")

df_train_rul = df_train.withColumn(
    "max_cycle",
    F.max("time_in_cycles").over(ventana_unit)
)

# RUL = último ciclo del motor - ciclo actual

df_train_rul = df_train_rul.withColumn(
    "RUL",
    F.col("max_cycle") - F.col("time_in_cycles")
)

# ---------------------------------------------------------
# RUL capado (para evitar que valores muy altos dominen el training
# ---------------------------------------------------------

RUL_CAP = 125

df_train_rul = df_train_rul.withColumn(
     "RUL_capado",
     F.when(F.col("RUL") > RUL_CAP, RUL_CAP).otherwise(F.col("RUL"))
)
# ---------------------------------------------------------
# 6. Eliminar columna auxiliar
# ---------------------------------------------------------
df_train_rul = df_train_rul.drop("max_cycle")
df_train_rul.select("RUL", "RUL_capado").show()

+---+----------+
|RUL|RUL_capado|
+---+----------+
|191|       125|
|190|       125|
|189|       125|
|188|       125|
|187|       125|
|186|       125|
|185|       125|
|184|       125|
|183|       125|
|182|       125|
|181|       125|
|180|       125|
|179|       125|
|178|       125|
|177|       125|
|176|       125|
|175|       125|
|174|       125|
|173|       125|
|172|       125|
+---+----------+
only showing top 20 rows



## Calculo el RUL para TEST (último ciclo del motor - ciclo actual) + RUL del dataset RUL_FD001.txt
**El cambio de fórmula se debe a que en el dataset TEST los motores no llegan al fallo, y el dataset adicional muestra los ciclos restantes hasta el fallo.**

In [23]:
# Cálculo del RUL para TEST
df_test2 = df_test.join(df_rul, on="unit_number", how="left")
df_test_rul = df_test2.withColumn(
    "max_cycle",
    F.max("time_in_cycles").over(ventana_unit)
)

# RUL = último ciclo del motor - ciclo actual + ciclos restantes
df_test_rul = df_test_rul.withColumn(
    "RUL",
    F.col("max_cycle") - F.col("time_in_cycles") + F.col("RUL_final")
)

# ---------------------------------------------------------
# RUL capado (para evitar que valores muy altos dominen el training
# ---------------------------------------------------------

RUL_CAP = 125

df_test_rul = df_test_rul.withColumn(
     "RUL_capado",
     F.when(F.col("RUL") > RUL_CAP, RUL_CAP).otherwise(F.col("RUL"))
)
# ---------------------------------------------------------
# 6. Eliminar columnas auxiliares
# ---------------------------------------------------------
df_test_rul = df_test_rul.drop("max_cycle")
df_test_rul = df_test_rul.drop("RUL_final")
df_test_rul.select("RUL", "RUL_capado").show()

+---+----------+
|RUL|RUL_capado|
+---+----------+
|142|       125|
|141|       125|
|140|       125|
|139|       125|
|138|       125|
|137|       125|
|136|       125|
|135|       125|
|134|       125|
|133|       125|
|132|       125|
|131|       125|
|130|       125|
|129|       125|
|128|       125|
|127|       125|
|126|       125|
|125|       125|
|124|       124|
|123|       123|
+---+----------+
only showing top 20 rows



In [26]:
# Dataframe final de train
df_train_rul.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("/opt/spark-data/processed/TRAIN_FD001_RUL")

Py4JJavaError: An error occurred while calling o310.csv.
: java.io.IOException: Failed to rename DeprecatedRawLocalFileStatus{path=file:/opt/spark-data/processed/TRAIN_FD001_RUL/_temporary/0/task_2026051617001076852077208791924_0110_m_000000/part-00000-c37e5db9-56db-44d2-8d92-4d1738311847-c000.csv; isDirectory=false; length=3284682; replication=1; blocksize=33554432; modification_time=1778950810519; access_time=1778950810519; owner=; group=; permission=rw-rw-rw-; isSymlink=false; hasAcl=false; isEncrypted=false; isErasureCoded=false} to file:/opt/spark-data/processed/TRAIN_FD001_RUL/part-00000-c37e5db9-56db-44d2-8d92-4d1738311847-c000.csv
	at org.apache.hadoop.mapreduce.lib.output.FileOutputCommitter.mergePaths(FileOutputCommitter.java:477)
	at org.apache.hadoop.mapreduce.lib.output.FileOutputCommitter.mergePaths(FileOutputCommitter.java:490)
	at org.apache.hadoop.mapreduce.lib.output.FileOutputCommitter.commitJobInternal(FileOutputCommitter.java:405)
	at org.apache.hadoop.mapreduce.li

In [27]:
# Dataframe final de test
df_test_rul.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("/opt/spark-data/processed/TEST_FD001_RUL")

Py4JJavaError: An error occurred while calling o314.csv.
: java.io.IOException: Failed to rename DeprecatedRawLocalFileStatus{path=file:/opt/spark-data/processed/TEST_FD001_RUL/_temporary/0/task_202605161700148516702185543067554_0114_m_000000/part-00000-fc008d97-a0bd-41a2-bc73-bbd3d2bb80e4-c000.csv; isDirectory=false; length=4291270; replication=1; blocksize=33554432; modification_time=1778950814510; access_time=1778950814510; owner=; group=; permission=rw-rw-rw-; isSymlink=false; hasAcl=false; isEncrypted=false; isErasureCoded=false} to file:/opt/spark-data/processed/TEST_FD001_RUL/part-00000-fc008d97-a0bd-41a2-bc73-bbd3d2bb80e4-c000.csv
	at org.apache.hadoop.mapreduce.lib.output.FileOutputCommitter.mergePaths(FileOutputCommitter.java:477)
	at org.apache.hadoop.mapreduce.lib.output.FileOutputCommitter.mergePaths(FileOutputCommitter.java:490)
	at org.apache.hadoop.mapreduce.lib.output.FileOutputCommitter.commitJobInternal(FileOutputCommitter.java:405)
	at org.apache.hadoop.mapreduce.li

**No he podido averiguar como solucionar el error de escritura, pero igualmente lo pasa a csv.**

In [28]:
# Compruebo que se han guardado correctamente
df_train_final = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("sep", ",")
    .csv("/opt/spark-data/processed/TRAIN_FD001_RUL.csv")
)
df_train_final.show()
df_test_final = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("sep", ",")
    .csv("/opt/spark-data/processed/TEST_FD001_RUL.csv")
)
df_test_final.show()

+-----------+--------------+-------------------+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+--------------------+---+----------+
|unit_number|time_in_cycles|   setting_1_scaled|    setting_2_scaled|     sensor_2_scaled|     sensor_3_scaled|     sensor_4_scaled|    sensor_7_scaled|     sensor_8_scaled|     sensor_9_scaled|    sensor_11_scaled|    sensor_12_scaled|    sensor_13_scaled|    sensor_14_scaled|    sensor_15_scaled|   sensor_17_scaled|    sensor_20_scaled|    sensor_21_scaled|RUL|RUL_capado|
+-----------+--------------+-------------------+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+--------------------+--------------------+--------------------+----------

In [29]:
spark.stop()